# Day 07 · 10 minute map — The Doors, Right Now

**Question.** Which accessible 'L' stations could a rider who cannot take the stairs not use at the
moment of the pull, because an elevator was out of service, and who lost the step-free walk to a
train?

**The 10-minute rule.** The challenge allows no more than 10 minutes to make the map. Everything that
does not depend on the moment is prepared beforehand: the station list (dated snapshot), the
step-free walk network and the base walkshed (cached), and the renderer. The timed run is
sections 2 to 8: one live pull of CTA's customer alerts, the analysis, and `plate.json`. The run's
start, pull and finish times are written to the plate.

**Approach.** Take the elevator alerts from CTA's Customer Alerts API (no key). Match each alert to
the platforms (stops) of its station in the CTA stop list. An outage that leaves **no** step-free way
to any platform takes the station out of the step-free network; an outage at **one** platform leaves
one direction. Rerun Day 03's step-free half-mile walk without the stations that lost every platform,
and count the residents (2020 Census blocks) who lose it.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings; run clock starts |
| P. Prepared | Stop snapshot, blocks, step-free network and base walkshed (cached; not timed) |
| 2. Pull | Live CTA alerts (`LIVE = True`) or a saved pull |
| 3. Validate | Feed structure, one station per alert, stations known |
| 4. Transform | Alert text → affected platforms → scope (station / platform) |
| 5. Analyze | Residents who lose the step-free half mile; outage ages; the hourly log |
| 6. Polygons | Step-free walkshed now, lost, one direction only |
| 7. QA view | Quick map and table |
| 8. Export | `plate.json` with the data ledger and the run times |

Run from this folder: `uv run jupyter nbconvert --execute --to notebook --inplace day07.ipynb`.
For the timed run set `LIVE = True`; otherwise the newest saved pull (or `PINNED`) is used, so the
published plate can be rebuilt exactly. The hourly log is `uv run python -m atlas.live data/log --elevators`
(run from the repository root with the day's path).

## 1. Setup

In [ ]:
from __future__ import annotations

import html
import json
import logging
import os
import pickle
import re
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

from atlas import Ledger, live, snapshot, to_address, walk, write_plate

RUN_STARTED = datetime.now(live.CHICAGO)
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('osmnx').setLevel(logging.WARNING)
log = logging.getLogger('day07')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
LIVE = os.environ.get('DAY07_LIVE') == '1'  # the timed run (run.sh) pulls CTA alerts now
PINNED = None     # e.g. 'cta_alerts_2026-11-08T1641.json'; None = newest saved pull
REFRESH_STOPS = False  # True = new stop snapshot from the Chicago Data Portal (prepared, not timed)

DAY = 7
STOPS_ID = '8pix-ypme'  # CTA - System Information - List of 'L' Stops
STOPS_NAME = "CTA - System Information - List of 'L' Stops"
STOPS_URL = f'https://data.cityofchicago.org/d/{STOPS_ID}'
ALERTS_NAME = 'CTA Customer Alerts API'
ALERTS_DOCS = 'https://www.transitchicago.com/developers/alerts/'

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
LOG_DIR = DAY_DIR / 'data' / 'log'
PROC_DIR = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
OSM_DIR = REPO / walk.OSM_DIR
BLOCKS_CSV = REPO / 'reference' / 'chicago-blocks-2020.csv'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
assert (DAY_DIR / 'day07.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

LINES = ('Red', 'Blue', 'Brown', 'Green', 'Orange', 'Purple', 'Pink', 'Yellow')
LINE_FLAG = {'Red': 'red', 'Blue': 'blue', 'Brown': 'brn', 'Green': 'g', 'Orange': 'o',
             'Purple': 'p', 'Pink': 'pnk', 'Yellow': 'y'}
DAY03_COUNTS = {'total': 144, 'accessible': 106}  # Plates I and III, same dataset

# Walk settings, as Day 03
WALK_M = walk.HALF_MILE_M
NETWORK_BUFFER_M = 1609.344
MAX_SNAP_M = 150
FRONTAGE_M = 40
MIN_HOLE_M2 = 20_000
TIME_LIMIT_MIN = 10

# Direction names in alerts that the stop list spells differently
ALIASES = {'forest park': 'forest pk', 'ohare': "o'hare"}
# Hand decisions for alerts the text rule cannot read, keyed by AlertId: {'stops': [stop_id, ...]}
OVERRIDES: dict[str, dict] = {}

## P. Prepared (not timed)

These inputs do not depend on the moment: the stop list (dated snapshot), blocks and city limits from
`reference/`, the step-free walk network (Day 03's rules) and the base step-free distances. The
network and the base are cached in `data/processed/`, so the timed run only reads them.

In [ ]:
STOP_COLS = 'stop_id,direction_id,stop_name,map_id,station_descriptive_name,ada,' \
            'red,blue,g,brn,p,y,pnk,o,location'
stops_raw, stops_pulled = snapshot(STOPS_ID, RAW_DIR, 'l_stops', refresh=REFRESH_STOPS, select=STOP_COLS)
blocks = pd.read_csv(BLOCKS_CSV, dtype={'geoid': str})
city = shapely.union_all(gpd.read_file(BOUNDARY_RAW).to_crs(walk.CRS).geometry)
boundary_pulled = BOUNDARY_RAW.stem.rsplit('_', 1)[-1]

truth = lambda s: s.astype(str).str.lower().str.strip().eq('true')
stops = stops_raw.assign(
    stop_id=stops_raw['stop_id'].astype(str), map_id=stops_raw['map_id'].astype(str),
    ada=truth(stops_raw['ada']),
    lat=pd.to_numeric(stops_raw['location.latitude']), lon=pd.to_numeric(stops_raw['location.longitude']),
    **{f: truth(stops_raw[f]) for f in LINE_FLAG.values()},
)
PAREN = re.compile(r'\((?P<inside>[^)]*)\)\s*$')
stops['dir_name'] = stops['stop_name'].str.extract(r'\(([^)]*)\)\s*$', expand=False).fillna('')


def parse_lines(name: str) -> list[str]:
    m = PAREN.search(name)
    inside = m.group('inside') if m else ''
    return [line for line in LINES if re.search(rf'\b{line}\b', inside)]


stations = (
    stops.groupby('map_id', as_index=False)
    .agg(name=('station_descriptive_name', 'first'), ada=('ada', 'all'),
         lat=('lat', 'mean'), lon=('lon', 'mean'), n_stops=('stop_id', 'size'))
)
stations['name'] = stations['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
stations['short'] = stations['name'].map(lambda s: PAREN.sub('', s).strip())
stations['lines'] = stations['name'].map(parse_lines)
stations['x'], stations['y'] = to_address(stations['lat'], stations['lon'])
counts = {'total': len(stations), 'accessible': int(stations['ada'].sum())}
log.info('Stations %s (Day 03: %s); stop snapshot %s', counts, DAY03_COUNTS, stops_pulled)
assert counts == DAY03_COUNTS, 'Station counts changed since Plate III; explain before publishing'


def utm(lat, lon) -> np.ndarray:
    pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs='EPSG:4326').to_crs(walk.CRS)
    return np.column_stack([pts.x, pts.y])


st_xy = utm(stations['lat'], stations['lon'])
bl_xy = utm(blocks['lat'], blocks['lon'])
stations['in_city'] = shapely.contains_xy(city, st_xy[:, 0], st_xy[:, 1])
ids = stations['map_id'].to_numpy()
ada = stations['ada'].to_numpy()
P = int(blocks['pop'].sum())

In [ ]:
# Step-free network and base walkshed, cached (building them takes about a minute)
pbf, osm_date = walk.latest_pbf(OSM_DIR)
cache = PROC_DIR / f'stepfree-{osm_date.isoformat()}-{stops_pulled.isoformat()}.pkl'
if cache.exists():
    with cache.open('rb') as f:
        prep = pickle.load(f)
else:
    area = gpd.GeoSeries([city.buffer(NETWORK_BUFFER_M)], crs=walk.CRS).to_crs('EPSG:4326').iloc[0]
    xml = walk.extract(pbf, area, OSM_DIR / f'chicago-walk-{osm_date.isoformat()}.osm')
    E = walk.Edges.of(walk.main_component(walk.step_free(walk.load(xml))))
    d, k, off = walk.reach(E, list(ids[ada]), st_xy[ada], limit=WALK_M * 1.5, max_snap=MAX_SNAP_M)
    bd, bk = walk.point_distance(E, d, k, bl_xy)
    shed = walk.shed(walk.reached(E, d, k, WALK_M).geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
    prep = {'E': E, 'base_d': bd, 'base_k': bk, 'base_shed': shed}
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    with cache.open('wb') as f:
        pickle.dump(prep, f)
E_free = prep['E']
blocks['d0'], blocks['k0'] = prep['base_d'], prep['base_k']
shed_base = prep['base_shed']
base_near = int(blocks.loc[blocks['d0'] <= WALK_M, 'pop'].sum())
log.info('Base: %s residents (%.1f%%) within a step-free half mile of an accessible station',
         f'{base_near:,}', 100 * base_near / P)
PREPARED = datetime.now(live.CHICAGO)

In [ ]:
# The 'L' track, drawn under the stations: CTA's GTFS rail shapes carrying at least a tenth of their
# line's trips, simplified (as Plates II and V)
import zipfile
from atlas import gtfs
gtfs.RAW_DIR = REPO / gtfs.RAW_DIR  # the shared feed folder (the module's path is relative to the repo root)
GTFS_ZIP, gtfs_date = gtfs.feed()
RAIL_NAMES = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}
read = lambda f, **kw: pd.read_csv(zipfile.ZipFile(GTFS_ZIP).open(f), dtype=str, skipinitialspace=True, **kw)
rail_routes = read('routes.txt').query('route_type == "1"')
rail_trips = read('trips.txt', usecols=['route_id', 'shape_id'])
rail_counts = rail_trips[rail_trips['route_id'].isin(rail_routes['route_id'])].groupby(['route_id', 'shape_id']).size()
rail_counts = rail_counts[rail_counts >= 0.1 * rail_counts.groupby(level=0).transform('sum')]
rail_paths = gtfs.shape_paths(GTFS_ZIP, set(rail_counts.index.get_level_values(1)))
track = [{'line': RAIL_NAMES[rid],
          'path': [[round(x), round(y)] for x, y in shapely.simplify(shapely.LineString(rail_paths[sid][['x', 'y']].to_numpy()), 10).coords]}
         for rid, sid in rail_counts.index]
assert {t['line'] for t in track} == set(LINES), 'A line has no rail shape'
log.info('Rail shapes (GTFS feed of %s): %d on %d lines', gtfs_date, len(track), len(LINES))

## 2. Pull (timed from here)

`LIVE = True` pulls the active alerts now and saves them as `data/raw/cta_alerts_<time>.json`.
Otherwise the pinned or newest saved pull is read. The hourly log in `data/log/` (elevator alerts
only) is read for context.

In [ ]:
TIMED_FROM = datetime.now(live.CHICAGO)
if LIVE:
    feed, pulled_at = live.pull_cta_alerts(RAW_DIR)
else:
    path = RAW_DIR / PINNED if PINNED else live.snapshots(RAW_DIR, 'cta_alerts')[-1]
    feed, pulled_at = live.read(path)
alerts_all = feed['CTAAlerts'].get('Alert') or []
alerts_all = [alerts_all] if isinstance(alerts_all, dict) else alerts_all
elev = live.elevator_alerts(feed)
log.info('Pull of %s: %d active alerts, %d elevator alerts', pulled_at.isoformat(), len(alerts_all), len(elev))

log_files = live.snapshots(LOG_DIR, 'cta_elevators')
log_pulls = [live.read(p) for p in log_files]
log.info('Hourly log: %d pulls, %s to %s', len(log_pulls),
         log_pulls[0][1].isoformat() if log_pulls else '-', log_pulls[-1][1].isoformat() if log_pulls else '-')

## 3. Validate

In [ ]:
def services(alert: dict) -> list[dict]:
    s = (alert.get('ImpactedService') or {}).get('Service') or []
    return [s] if isinstance(s, dict) else s


def text_of(value) -> str:
    if isinstance(value, dict):
        value = value.get('#cdata-section', '')
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', html.unescape(value or ''))).strip()


checks = {
    'feed reports no error': str(feed['CTAAlerts'].get('ErrorCode', '0')) in ('0', 'None'),
    'alerts have ids': all(a.get('AlertId') for a in alerts_all),
    'each elevator alert names one station': all(sum(s['ServiceType'] == 'T' for s in services(a)) == 1 for a in elev),
    'elevator stations are known': all(s['ServiceId'] in set(ids) for a in elev for s in services(a) if s['ServiceType'] == 'T'),
    'elevator alerts have a start': all(a.get('EventStart') for a in elev),
    'pull is not in the future': pulled_at <= datetime.now(live.CHICAGO),
}
for name, ok in checks.items():
    log.info('%-40s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

## 4. Transform

**Which platforms an alert closes.** The alert's own words before "at <station>" describe the
elevator ("The Kimball- and Linden-bound platform elevator", "The elevator to/from street", "The
elevator at 203 N. LaSalle (Brown and Harlem-bound Green Line platform …)"). Each stop of the station
in the CTA stop list is one platform direction, named by its terminals ("Chicago (Kimball-Linden-bound)")
and flagged by line. A stop is affected when the description names one of its terminals, or names a
line without a direction that the stop serves. A street elevator, or an elevator described without a
platform, is taken to serve every platform (CTA names the platform when only one is affected).
**Scope:** *station* when every stop of the station is affected (no step-free way to any train),
*platform* otherwise. Duplicate alerts for the same elevator are merged; the earliest start counts.

In [ ]:
def norm(s: str) -> str:
    s = s.lower().replace('’', "'").replace('-', ' ').strip()
    return ALIASES.get(s, s)


BOUND = re.compile(r"((?:[A-Z][\w'.]*(?:\s[A-Z][\w'.]*)?-\s*(?:and\s+|/\s*)?)+)bound")


def describe(alert: dict, station: str) -> str:
    """The alert's description of the elevator: its short text up to 'at <station>'."""
    t = text_of(alert.get('ShortDescription')).replace('- bound', '-bound')
    i = t.rfind(f' at {station}')
    return (t[:i] if i >= 0 else t).strip()


def stop_terms(dir_name: str) -> set[str]:
    return {norm(p) for p in re.split(r'[-/]', dir_name.replace('-bound', '').replace(' bound', '')) if p.strip()}


def affected(alert: dict, sid: str, station: str) -> tuple[list[str], str]:
    """Stop ids the alert closes, and the rule that decided it."""
    if alert['AlertId'] in OVERRIDES:
        return OVERRIDES[alert['AlertId']]['stops'], 'override'
    desc = describe(alert, station)
    st = stops[stops['map_id'] == sid]
    if re.search(r'\bstreet\b', desc, re.I) and not re.search(r'\bplatform\b', desc, re.I):
        return st['stop_id'].tolist(), 'street elevator'
    terms = {norm(n) for m in BOUND.finditer(desc) for n in re.split(r'-\s*(?:and\s+|/\s*)?', m.group(1)) if n.strip()}
    plain_lines = [l for l in LINES if re.search(rf'(?<!bound )\b{l}\b', desc)]
    mask = st['dir_name'].map(lambda d: bool(stop_terms(d) & terms))
    if plain_lines:
        mask |= st[[LINE_FLAG[l] for l in plain_lines]].any(axis=1)
    hit = st[mask]
    if terms or plain_lines:
        assert len(hit), f'Alert {alert["AlertId"]} names {terms or plain_lines} but no stop of {station} matches: add an override'
        return hit['stop_id'].tolist(), 'named platform'
    return st['stop_id'].tolist(), 'elevator without a platform named'


rows = []
for a in elev:
    t = next(s for s in services(a) if s['ServiceType'] == 'T')
    sid, sname = t['ServiceId'], t['ServiceName']
    hit, rule = affected(a, sid, sname)
    full = text_of(a.get('FullDescription'))
    ret = re.search(r'return to service on (?:\w+day,? )?(\w+ \d{1,2})(?:st|nd|rd|th)?,? (\d{4})', full)
    reason = re.search(r'out-of-service,? (?:due to|for) ([^.]+)\.', full)
    rows.append({
        'alert': a['AlertId'], 'map_id': sid, 'description': describe(a, sname), 'rule': rule,
        'stops': tuple(sorted(hit)), 'start': pd.Timestamp(a['EventStart']).tz_localize(live.CHICAGO),
        'estimatedReturn': (pd.to_datetime(f'{ret.group(1)} {ret.group(2)}').date().isoformat() if ret else None),
        'reason': reason.group(1).strip() if reason else None,
    })
raw_out = pd.DataFrame(rows)

# one row per elevator: same station, same platforms
outages = (
    raw_out.sort_values('start')
    .groupby(['map_id', 'stops'], as_index=False)
    .agg(alerts=('alert', list), description=('description', 'first'), rule=('rule', 'first'),
         start=('start', 'first'), estimatedReturn=('estimatedReturn', 'max'), reason=('reason', 'first'))
    .merge(stations[['map_id', 'short', 'lines', 'ada', 'n_stops', 'in_city']], on='map_id')
)
outages['n_hit'] = outages['stops'].map(len)
# a station is cut off when the union of its outages covers every stop
cover = outages.groupby('map_id')['stops'].agg(lambda s: len(set().union(*s)))
outages['station_hit'] = outages['map_id'].map(cover)
outages['scope'] = np.where(outages['station_hit'] >= outages['n_stops'], 'station', 'platform')
outages['ageH'] = (pd.Timestamp(pulled_at) - outages['start']).dt.total_seconds() / 3600
outages = outages.sort_values(['scope', 'start'], ascending=[False, True], ignore_index=True)
if not outages['ada'].all():  # an elevator at a station that is not accessible anyway: shown, not counted
    log.warning('Outage at a non-accessible station: %s', outages.loc[~outages['ada'], 'short'].tolist())
outages[['short', 'lines', 'description', 'rule', 'stops', 'n_stops', 'scope', 'start', 'ageH', 'estimatedReturn', 'alerts']]

## 5. Analyze

In [ ]:
cut = outages.loc[(outages['scope'] == 'station') & outages['ada'], 'map_id'].unique()
part = outages.loc[(outages['scope'] == 'platform') & outages['ada'], 'map_id'].unique()


def without(drop) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Step-free distances (nodes and blocks) with these accessible stations removed."""
    keep = ada & ~np.isin(ids, drop)
    d, k, _ = walk.reach(E_free, list(ids[keep]), st_xy[keep], limit=WALK_M * 1.5, max_snap=MAX_SNAP_M)
    bd, bk = walk.point_distance(E_free, d, k, bl_xy)
    return d, k, bd, bk


d1, k1, blocks['d1'], _ = without(cut)               # now: stations with no step-free platform removed
d2, k2, blocks['d2'], _ = without([*cut, *part])     # upper bound: every station with an outage removed
had = blocks['d0'] <= WALK_M
blocks['now'] = np.select([blocks['d1'] <= WALK_M, had], ['kept', 'lost'], 'none')
lost = blocks['now'] == 'lost'
one_dir = had & ~lost & (blocks['d2'] > WALK_M)  # still near a station, but only one direction step-free

by_station = (blocks[lost].groupby('k0')[['pop', 'amb']].sum()
              .reindex(cut, fill_value=0).rename(columns={'pop': 'lost', 'amb': 'lostAmb'}))
by_station_1d = (blocks[one_dir].groupby('k0')[['pop']].sum()
                 .reindex(part, fill_value=0).rename(columns={'pop': 'oneDirection'}))
outages = (outages.merge(by_station, left_on='map_id', right_index=True, how='left')
                  .merge(by_station_1d, left_on='map_id', right_index=True, how='left'))
outages[['lost', 'lostAmb', 'oneDirection']] = outages[['lost', 'lostAmb', 'oneDirection']].fillna(0)

summary = {
    'residents': P,
    'accessibleStations': counts['accessible'],
    'elevatorAlerts': len(elev),
    'elevators': len(outages),
    'stationsWithOutage': int(outages['map_id'].nunique()),
    'stationsCut': len(cut),
    'stationsOneDirection': len(part),
    'stationsCutOutsideCity': int((~stations.set_index('map_id').loc[cut, 'in_city']).sum()) if len(cut) else 0,
    'baseNear': base_near,
    'baseNearShare': base_near / P,
    'nowNear': int(blocks.loc[blocks['now'] == 'kept', 'pop'].sum()),
    'lost': int(blocks.loc[lost, 'pop'].sum()),
    'lostAmbulatory': float(blocks.loc[lost, 'amb'].sum()),
    'oneDirection': int(blocks.loc[one_dir, 'pop'].sum()),
    'oldestDays': float(outages['ageH'].max() / 24),
    'startedToday': int((outages['start'].dt.date == pulled_at.date()).sum()),
}
assert summary['lost'] == int(outages['lost'].sum()), 'Lost residents must add up by station'
log.info('Summary: %s', {k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items()})
pd.Series(summary)

In [ ]:
# The hourly log: every elevator seen, first and last time, so the plate can say how much changes.
seen = {}
for data, at in log_pulls + [({'Elevator': elev}, pulled_at)]:
    for a in data['Elevator']:
        t = next((s for s in services(a) if s['ServiceType'] == 'T'), None)
        if not t:
            continue
        key = (t['ServiceId'], describe(a, t['ServiceName']).lower())
        first, last = seen.get(key, (at, at))
        seen[key] = (min(first, at), max(last, at))
history = {
    'pulls': len(log_pulls),
    'from': log_pulls[0][1].isoformat() if log_pulls else pulled_at.isoformat(),
    'to': max([p[1] for p in log_pulls] + [pulled_at]).isoformat(),
    'elevatorsSeen': len(seen),
    'stationsSeen': len({k[0] for k in seen}),
}
log.info('History: %s', history)

In [ ]:
# The finding printed on the plate, generated from the data.
clock = lambda t: t.strftime('%-I:%M ') + ('a.m.' if t.hour < 12 else 'p.m.')
day_s = lambda t: t.strftime('%-d %B %Y')
num = lambda n: {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven',
                 8: 'eight', 9: 'nine', 10: 'ten'}.get(n, f'{n:,}')
hundreds = lambda n: f'{round(n, -2):,.0f}'
stn = lambda n: 'station' if n == 1 else 'stations'
FINDING = (
    f'At {clock(pulled_at)} on {day_s(pulled_at)}, {num(summary["stationsCut"])} accessible '
    f'{stn(summary["stationsCut"])} had no working elevator to any train, and '
    f'{num(summary["stationsOneDirection"])} more had lost one platform. '
    f'About {hundreds(summary["lost"])} Chicagoans lost their step-free half-mile walk to the ‘L’.'
)
print(FINDING)

## 6. Polygons

As Day 03: reached streets widened by `FRONTAGE_M`, merged and clipped to the city. *Now* is the
step-free walkshed without the stations that lost every platform; *lost* is the base walkshed minus
*now*; *one direction* is the part of *now* that depends on a station with one platform out.

In [ ]:
shed_now = walk.shed(walk.reached(E_free, d1, k1, WALK_M).geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
shed_up = walk.shed(walk.reached(E_free, d2, k2, WALK_M).geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
shed_lost = shed_base.difference(shed_now)
shed_1d = shed_now.difference(shed_up)
SQ_MI = 2_589_988.11
area = {'baseSqMi': shed_base.area / SQ_MI, 'nowSqMi': shed_now.area / SQ_MI,
        'lostSqMi': shed_lost.area / SQ_MI, 'oneDirectionSqMi': shed_1d.area / SQ_MI}
log.info('Areas (sq mi): %s', {k: round(v, 2) for k, v in area.items()})
rings = {'now': walk.to_rings(shed_now), 'lost': walk.to_rings(shed_lost), 'oneDirection': walk.to_rings(shed_1d)}
log.info('Vertices: %s', {k: sum(len(r) for p in v for r in p) for k, v in rings.items()})

## 7. QA view

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
gpd.GeoSeries([city], crs=walk.CRS).plot(ax=ax, color='0.92')
gpd.GeoSeries([shed_now], crs=walk.CRS).plot(ax=ax, color='0.7')
for g, hatch in ((shed_lost, '////'), (shed_1d, '..')):
    if not g.is_empty:
        gpd.GeoSeries([g], crs=walk.CRS).plot(ax=ax, facecolor='none', hatch=hatch, edgecolor='0.1', linewidth=0)
ax.scatter(*st_xy[ada].T, s=6, color='0.2', zorder=3)
for r in outages.itertuples():
    p = st_xy[np.flatnonzero(ids == r.map_id)[0]]
    ax.scatter(*p, s=60, facecolor='none', edgecolor='red' if r.scope == 'station' else 'orange', lw=1.5, zorder=4)
    ax.annotate(r.short, p, xytext=(5, 3), textcoords='offset points', fontsize=7)
minx, miny, maxx, maxy = city.bounds
ax.set(xlim=(minx, maxx + 2000), ylim=(miny, maxy + 4000),
       title=f'Elevators out at {pulled_at:%Y-%m-%d %H:%M}: station (red), platform (orange)')
ax.set_axis_off()
plt.show()
outages[['short', 'scope', 'rule', 'description', 'lost', 'oneDirection', 'ageH']].round(1)

## 8. Export

`plate.json`: ledger, finding, summary, outages, stations, polygons and the run clock. Keys are
camelCase. The stop list is the only dataset that is redistributed (CSV); the alerts are summarized.

In [ ]:
FINISHED = datetime.now(live.CHICAGO)
elapsed_min = (FINISHED - TIMED_FROM).total_seconds() / 60
if LIVE:
    assert elapsed_min < TIME_LIMIT_MIN, f'The timed run took {elapsed_min:.1f} min'

ledger = Ledger(
    source=ALERTS_NAME,
    dataset_id='alerts.aspx (activeonly)',
    url=ALERTS_DOCS,
    pulled=pulled_at.date(),
    units=f'{len(elev)} elevator alerts at {summary["stationsWithOutage"]} stations; {P:,} residents (2020 Census) in {len(blocks):,} blocks',
    method=[
        f'One pull of CTA’s active customer alerts at {pulled_at:%H:%M} Chicago time; impact "Elevator Status"',
        'Alert text matched to the platforms (stops) of its station in the CTA stop list',
        'Station scope: no step-free way to any platform; platform scope: one direction lost',
        f'Half-mile step-free walk on the OpenStreetMap network (extract of {osm_date}), as Plate III',
    ],
    caveats=[
        'Alerts describe elevators in words; the platform match is a rule, checked by hand on each run',
        'An outage is counted from the alert’s start; CTA does not publish when it is fixed, only that the alert ends',
        'Ambulatory difficulty: ACS 2020-2024 tract estimates spread over blocks by population',
        'Residents outside Chicago (Evanston, Skokie, Oak Park, Forest Park) are not counted',
    ],
)

iso = lambda t: pd.Timestamp(t).isoformat()
text_or_none = lambda v: v if isinstance(v, str) else None  # pandas turns missing text into NaN
r1 = lambda v: round(float(v), 1)
payload = {
    'source': {'name': ALERTS_NAME, 'url': ALERTS_DOCS},
    'stops': {'name': STOPS_NAME, 'id': STOPS_ID, 'url': STOPS_URL, 'pulled': stops_pulled.isoformat()},
    'osm': {'extract': osm_date.isoformat(), 'source': 'Geofabrik, Illinois'},
    'gtfs': {'feedDate': gtfs_date.isoformat(), 'url': gtfs.FEED_URL},
    'rail': track,
    'run': {'started': iso(TIMED_FROM), 'pulledAt': iso(pulled_at), 'finished': iso(FINISHED),
            'minutes': round(elapsed_min, 2), 'live': LIVE, 'limitMinutes': TIME_LIMIT_MIN},
    'walkM': round(WALK_M, 1),
    'summary': {**summary, **area},
    'history': history,
    'outages': [
        {'id': r.map_id, 'short': r.short, 'lines': r.lines, 'scope': r.scope, 'description': r.description,
         'stops': list(r.stops), 'nStops': int(r.n_stops), 'alerts': r.alerts, 'start': iso(r.start),
         'ageHours': r1(r.ageH), 'estimatedReturn': text_or_none(r.estimatedReturn), 'reason': text_or_none(r.reason),
         'inCity': bool(r.in_city), 'lost': int(r.lost), 'lostAmbulatory': r1(r.lostAmb),
         'oneDirection': int(r.oneDirection)}
        for r in outages.itertuples()
    ],
    'stations': [
        {'id': r.map_id, 'short': r.short, 'lines': r.lines, 'ada': bool(r.ada),
         'x': r1(r.x), 'y': r1(r.y), 'inCity': bool(r.in_city)}
        for r in stations.itertuples()
    ],
    'polygons': rings,
}
path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
written = json.loads(path.read_text())
assert {'run', 'summary', 'outages', 'stations', 'polygons', 'finding'} <= written.keys()
assert 'NaN' not in path.read_text(), 'NaN is not valid JSON for the site'
log.info('Wrote %s (%.1f KB); timed run %.2f min (live=%s)', path.relative_to(DAY_DIR),
         path.stat().st_size / 1024, elapsed_min, LIVE)

## Notes for the plate

- The finding is generated in section 5. Every number on the plate comes from `plate.json`.
- Check the section 4 table by eye on every run: the description, the platforms matched and the scope.
  Add an entry to `OVERRIDES` for any alert the rule cannot read.
- The hourly log (`data/log/`) gives context only; the plate shows the moment of the pull.